# 🖼️ Image Models - Validation Tests

## Overview

Validate **image generation and editing** through the Unified AI API (`/unified-ai`). All image providers are routed through a single OpenAI-style images surface and return OpenAI-shaped responses (`data[].b64_json`).

This notebook tests:
- **Backend Onboarding (FLUX & MAI)**: augment the azd-loaded `llm_backends_config` with dedicated `azure-flux` / `azure-mai` backends and redeploy the onboarding Bicep (`gpt-image` rides the existing `ai-foundry` backend)
- **Access Contract Provisioning**: Deploy an access contract via Bicep to obtain API credentials scoped to the image models
- **Deployment Discovery**: Verify the image models appear in `GET /unified-ai/deployments`
- **gpt-image generation (body model)**: `POST /unified-ai/v1/images/generations` with `model` in the body (Azure OpenAI / Foundry)
- **gpt-image generation (deployments path)**: `POST /unified-ai/openai/deployments/{model}/images/generations`
- **FLUX generation** (`azure-flux`, optional): native Black Forest Labs surface via `modelPath` slug
- **MAI generation** (`azure-mai`, optional): native Microsoft MAI surface
- **Missing-model error**: `POST /unified-ai/v1/images/edits` without `x-ai-model` returns `400 missing_model_parameter`
- **Image edit**: edit a generated image via `POST /unified-ai/openai/deployments/{model}/images/edits` (multipart)
- **RBAC enforcement**: a model outside the access contract's `allowedModels` returns `403`

> **Prerequisites:** An existing Citadel Governance Hub deployment with the Unified AI API enabled and at least one image model onboarded (see [LLM Backend Onboarding — Image Models](../bicep/infra/llm-backend-onboarding/README.md#image-models)).

### Shared workflow and safety

Run from the repository root or `validation`. Explicit `overrides` win over the
selected azd environment. Model operations are inferred automatically; optional `operation_overrides`
(or `VALIDATION_MODEL_CAPABILITIES`) replace inference for individual models.
Image operations use `image-generation` and `image-edit`; affinity uses `responses`.
Definitions and policy XML remain visible. Preview the generated **`.bicepparam`**
in one cell, optionally edit it, then deploy it in the following cell; the shared
deploy helper compiles the actual parameter file before submission.
Generated artifacts live under ignored `validation/.runs`. Tokens/keys stay in
memory. `run.finish()` raises for required FAIL/BLOCKED/empty runs. Cleanup is off
and can only delete resources absent before this run. Cloud execution has real cost
and side effects; local source validation is not a cloud feature verdict.

### 0️⃣ Initialize Notebook Variables

**Choose ONE initialization mode** by setting `init_from_azd`:

- `True` — autoload `governance_hub_resource_group` and `location` from your active `azd` environment.
- `False` — set explicit resource coordinates in `overrides`.

Set the image model names to match your onboarded backends, and enable the `test_*` flags for the providers you have configured.

### 0️⃣ Initialize Notebook Variables

Configure this notebook's environment, explicit feature opt-ins, and run-owned resources without deploying anything.

**Input:** The selected repository kernel; edit the configuration values below and provide required secrets through environment variables.

**Output:** Configuration globals for the following initialization step; no cloud deployment or test is performed.


### Automatic models and optional overrides

Leave `environment=None`, azd enabled, `operation_overrides=None`, and
`skipped_models=None` to use the selected deployment without editing model lists.
Operations default to name-based **assumptions**, verified by real requests:
`image`/`flux` → image generation/edit; otherwise `embedding`/`text` → embeddings;
otherwise `gpt` → chat and Responses; other names → chat. Matching ignores case.

Optional examples (replace example names with names in the loaded inventory):

```python
# Replace operations only for these models; other models retain automatic mapping.
operation_overrides = {"my-chat-model": ["chat", "responses"], "my-image-model": ["image-generation"]}
# Use operation_overrides = {} to discard stored azd operation overrides.
# Disable one model's tests with an empty override: {"my-model": []}.
skipped_models = ["mini", "legacy"]  # Case-insensitive substrings; backend deployment is unchanged.
overrides = {"resource_group": "rg-other-hub", "apim_name": "apim-other-hub"}  # Explicit azd-value overrides.
# To require a complete manually mapped catalogue: overrides["infer_operations"] = False.
```

Streaming is assumed for inferred chat models. Explicit mappings need `streaming`
when that scenario is selected. Responses GET/input-items returning HTTP 404 are warnings/SKIP for every model,
including explicit overrides, not successful retrieval. Other unsupported-operation
exceptions remain limited to inferred GPT-mini models; explicit mappings require
those non-404 retrieval checks. Authentication, throttling, and server failures
are never silently accepted.

### Scenario-specific input examples


`openai_image_override`, `flux_image_override`, `mai_image_override`, and
`edit_header_override` optionally select exact deployed model names; for example
`openai_image_override = "my-image-model"`. `None` selects by resolved operations.
`disallowed_image_override = "another-deployed-image-model"` selects a real
RBAC-denial control outside the access grant. Available native providers are selected automatically;
absent optional providers and negative controls are reported as SKIP.
To onboard additional native endpoints, provide `image_backend_overrides` using
the backend-contract shape (`backendId`, `backendType`, `endpoint`, `supportedModels`),
with deployment enabled by default. The complete azd inventory, aliases and global settings are retained;
there is no manual approval prerequisite. Generated images are run-owned multipart-edit inputs.

Run All discovers available OpenAI, FLUX and MAI image models automatically and provisions routing from the complete azd inventory. Source JSON is optional; APIM identity is discovered through the SDK. Missing provider/edit/negative-control model prerequisites are SKIP, not mandatory configuration. Keyword-excluded models remain deployed but are never selected for generation, edits or negative controls.


In [ ]:
from pathlib import Path
import sys
repo_root = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "azure.yaml").is_file() and (p / "shared" / "utils.py").is_file()), None)
if repo_root is None:
    raise RuntimeError("Open this repository and run from its root or validation directory.")
sys.path.insert(0, str(repo_root / "shared"))
import utils
with utils.notebook_step('0️⃣ Initialize Notebook Variables', inputs={'repository': str(repo_root if 'repo_root' in globals() else root)}, expected='Configuration globals for the following initialization step; no cloud deployment or test is performed.', namespace=globals()):
    import os, sys, json, time, uuid, base64, requests
    from pathlib import Path
    from apimtools import APIMClientTool
    from notebookdefaults import backend_source, source_catalog
    from validationtools import ValidationContext, ValidationRun, RequestTransport, ValidationBlocked
    from contracttools import write_parameters, deploy_contract, access_policy, preview
    from scenariotools import access_parameters, contract_identity, cleanup_products, policy_xml, completion_text, save_manifest

    init_from_azd = True
    environment = None  # Explicit azd environment, or the active environment.
    overrides = {}  # Explicit values win: resource_group/location/subscription_id/apim_name.
    # Supply capability metadata, not name patterns, e.g.
    # overrides["capabilities"] = {"your-deployed-model": ["chat", "responses"]}
    run_id = uuid.uuid4().hex[:12]
    cleanup_delete_contract = False

    import re, copy
    # Required primary image route; additional providers are explicit opt-ins.
    source_parameters_file = None  # Optional complete-source JSON override.
    openai_image_override = None
    flux_image_override = None
    mai_image_override = None
    edit_header_override = None
    disallowed_image_override = None
    test_openai_image = True
    test_flux = True
    test_mai = True
    run_backend_onboarding = True
    backend_configuration_complete = True  # Confirm full effective config before shared routing changes.
    image_backend_overrides = []  # Full explicitly provisioned image backend definitions, including modelPath/auth.
    image_prompt = "A photograph of a red fox in an autumn forest, cinematic lighting"
    save_images = True  # Run-owned PNG artifacts are also used by multipart edit examples.
    contract_business_unit = "Testing"
    contract_use_case_name = "ImageModels-" + run_id
    contract_environment = "DEV"

    # None uses azd operation overrides when present, otherwise automatic name-based mapping.
    # Example: {"my-chat-model": ["chat", "responses"], "my-image-model": ["image-generation"]}
    # An empty list for one model disables its tests; explicit lists replace that model's inferred operations.
    # A supplied dictionary replaces azd's operation map; {} resets it to automatic mapping.
    operation_overrides = None
    # None uses VALIDATION_SKIPPED_MODELS from azd (default []); matches substrings without case sensitivity.
    # Example: ["mini", "legacy"] skips any model containing either keyword in validation, not deployment.
    skipped_models = None


### 1️⃣ Verify Azure CLI and Connected Subscription

### 1️⃣ Verify Azure CLI and Connected Subscription

Resolve the selected deployment and validate this scenario's mandatory inputs before deployment or feature tests.

**Input:** Configuration from the previous cell, resource coordinates, inferred or explicitly overridden model operations, and enabled integration prerequisites.

**Output:** Validated scenario globals; missing inputs are aggregated by label and raise immediately, stopping Run All.


In [ ]:
with utils.notebook_step('1️⃣ Verify Azure CLI and Connected Subscription', inputs={'run_backend_onboarding': globals().get('run_backend_onboarding'), 'run_id': globals().get('run_id'), 'overrides': globals().get('overrides'), 'environment': globals().get('environment'), 'init_from_azd': globals().get('init_from_azd'), 'test_openai_image': globals().get('test_openai_image')}, expected='Validated scenario globals; missing inputs are aggregated by label and raise immediately, stopping Run All.', namespace=globals()):
    run = ValidationRun("image-models")
    context = ValidationContext.load(overrides=overrides, environment=environment, from_azd=init_from_azd,
        operation_overrides=operation_overrides, skipped_models=skipped_models)
    utils.require_inputs({f"gateway.{key}": context.config.get(key)
        for key in ("subscription_id", "resource_group", "location", "apim_name")})
    governance_hub_resource_group = context.config["resource_group"]
    location = context.config["location"]
    subscription_id = context.config["subscription_id"]
    apimClientTool = APIMClientTool(governance_hub_resource_group, apim_resource_name=context.config["apim_name"])
    apimClientTool.initialize_context(context.account, enumerate_subscriptions=False)
    source_parameters = backend_source(context, apimClientTool, source_parameters_file,
        overrides.get("backend_template_options"))
    context.catalog = source_catalog(context, source_parameters)
    llm_backends_config = source_parameters["llmBackendConfig"]
    run_dir = repo_root / "validation" / ".runs" / "image-models" / run_id
    transport = RequestTransport(timeout=120)

    def image_selection(label, enabled, override, backend_types, operation="image-generation"):
        if not enabled:
            run.skip(label, "Disabled explicitly.")
            return None
        eligible = [name for name in context.catalog.models_for(operation)
            if any(entry.name == name and entry.backend_type in backend_types for entry in context.catalog.entries)]
        if override is not None:
            context.catalog.select(operation, override=override)
            if override not in eligible:
                raise ValueError(f"{label}: override is not eligible for the selected image provider.")
            return override
        if not eligible:
            run.skip(label, f"No eligible {operation} model is configured for {sorted(backend_types)}.")
            return None
        return eligible[0]

    openai_image_model = image_selection("openai-generation", test_openai_image, openai_image_override,
        {"ai-foundry", "azure-openai"})
    flux_image_model = image_selection("flux-generation", test_flux, flux_image_override, {"azure-flux"})
    mai_image_model = image_selection("mai-generation", test_mai, mai_image_override, {"azure-mai"})
    test_openai_image = openai_image_model is not None
    test_flux = flux_image_model is not None
    test_mai = mai_image_model is not None
    allowed_models_list = list(dict.fromkeys(model for model in
        (openai_image_model, flux_image_model, mai_image_model) if model))
    image_ready = bool(allowed_models_list)
    edit_candidates = [model for model in allowed_models_list if model in context.catalog.models_for("image-edit")]
    edit_header_model = context.catalog.select("image-edit", override=edit_header_override) if edit_header_override else next(iter(edit_candidates), None)
    if edit_header_model and edit_header_model not in allowed_models_list:
        allowed_models_list.append(edit_header_model)
    if not edit_header_model:
        run.skip("image-edit-selection", "No eligible image-edit model is granted; generation coverage remains available.")
    disallowed_candidates = [model for model in context.catalog.models_for("image-generation") if model not in allowed_models_list]
    disallowed_image_model = context.catalog.select("image-generation", override=disallowed_image_override) if disallowed_image_override else next(iter(disallowed_candidates), None)
    if disallowed_image_model in allowed_models_list:
        raise ValueError("RBAC negative override must be deployed but outside the contract grants.")
    if not disallowed_image_model:
        run.skip("image-rbac-selection", "No eligible ungranted image model; RBAC negative coverage is optional.")
    utils.format_output("Image initialization", {"allowed_models": allowed_models_list,
        "edit_model": edit_header_model, "negative_model": disallowed_image_model,
        "scenario_ready": image_ready, "deployment_backends": len(llm_backends_config)}, status="info", part="RESULT")
    run_dir.mkdir(parents=True, exist_ok=True)


### 2️⃣ Initialize APIM Client Tool

Discover the Unified AI API and retrieve gateway configuration:

### 2️⃣ Initialize APIM Client Tool

Prepare the 2️⃣ Initialize APIM Client Tool state shown below for its subsequent feature-specific requests and assertions.

**Input:** `governance_hub_resource_group`

**Output:** `apimClientTool`, `unified_api`, `gateway_url`, `azure_endpoint`, `apim_resource_gateway_url`, `apim_name` are available to the following cells.


In [ ]:
with utils.notebook_step('2️⃣ Initialize APIM Client Tool', inputs={'governance_hub_resource_group': globals().get('governance_hub_resource_group')}, expected='`apimClientTool`, `unified_api`, `gateway_url`, `azure_endpoint`, `apim_resource_gateway_url`, `apim_name` are available to the following cells.', namespace=globals()):
    if not image_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if image_ready:
        unified_api = apimClientTool.discover_api_exact("unified-ai")
        gateway_url = str(apimClientTool.apim_resource_gateway_url).rstrip("/")
        azure_endpoint = gateway_url + "/"
        apim_resource_gateway_url = gateway_url
        apim_name = apimClientTool.apim_resource_name
        preexisting_products = {p.name for p in apimClientTool.client.product.list_by_service(
            governance_hub_resource_group, apim_name)}
        owned_products = set()


### 2️⃣B LLM Backend Contract Onboarding (FLUX & MAI image backends)

`gpt-image-1.5` works out of the box on your existing `ai-foundry` backend — it needs no special backend type. **FLUX** and **MAI** require dedicated `azure-flux` / `azure-mai` backends that point at the Foundry resource's native image surfaces (`/providers/blackforestlabs/v1/...` and `/mai/v1/images/...`).

This step augments the `llm_backends_config` loaded from your `azd` environment with:
- `gpt-image-1.5` added to the existing `ai-foundry` backend (only if missing)
- an `azure-flux` backend carrying the `modelPath` slug (e.g. `flux-2-pro`)
- an `azure-mai` backend (model travels in the request body)

then redeploys the LLM Backend Onboarding Bicep so the routing + `metadata-config` pick up the new image models. Skipped when `run_backend_onboarding = True`, or when neither FLUX nor MAI is enabled and `gpt-image` is already onboarded.

### 2️⃣B LLM Backend Contract Onboarding (FLUX & MAI image backends)

Generate the editable parameter artifact for 2️⃣B LLM Backend Contract Onboarding (FLUX & MAI image backends) and display the proposed configuration without deploying it.

**Input:** `run_backend_onboarding`, `image_backend_overrides`, `llm_backends_config`, `backend_configuration_complete`, `subscription_id`, `governance_hub_resource_group`

**Output:** A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.


In [ ]:
with utils.notebook_step('2️⃣B LLM Backend Contract Onboarding (FLUX & MAI image backends)', inputs={'run_backend_onboarding': globals().get('run_backend_onboarding'), 'image_backend_overrides': globals().get('image_backend_overrides'), 'llm_backends_config': globals().get('llm_backends_config'), 'backend_configuration_complete': globals().get('backend_configuration_complete'), 'subscription_id': globals().get('subscription_id'), 'governance_hub_resource_group': globals().get('governance_hub_resource_group')}, expected='A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.', namespace=globals()):
    if not image_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if image_ready:
        onboarding_params = None
        if run_backend_onboarding:
            onboarding_parameters = copy.deepcopy(source_parameters)
            onboard_cfg = onboarding_parameters["llmBackendConfig"]
            for backend in image_backend_overrides:
                backend_id = backend["backendId"]
                matches = [index for index, item in enumerate(onboard_cfg) if item["backendId"] == backend_id]
                if matches:
                    onboard_cfg[matches[0]] = copy.deepcopy(backend)
                else:
                    onboard_cfg.append(copy.deepcopy(backend))
            live_backends, _ = apimClientTool.get_backends()
            omitted = {item["name"] for item in live_backends} - {item["backendId"] for item in onboard_cfg}
            if omitted:
                raise ValueError(f"Complete source omitted live backends: {sorted(omitted)}")
            onboarding_params = write_parameters(repo_root / "bicep" / "infra" / "llm-backend-onboarding" / "main.bicep",
                onboarding_parameters, run_dir / "image-backends")
            utils.format_output("Image backend preview", {"parameters": preview(onboarding_params)},
                status="info", part="RESULT")
        else:
            run.skip("backend-onboarding", "Using existing image routing; deployment disabled explicitly.")


### 2️⃣B LLM Backend Contract Onboarding (FLUX & MAI image backends) — continuation 5

Deploy the reviewed artifact for 2️⃣B LLM Backend Contract Onboarding (FLUX & MAI image backends) — continuation 5, then inspect its real resource outputs and ownership.

**Input:** `onboarding_params`, `location`, `run_id`

**Output:** Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.


In [ ]:
with utils.notebook_step('2️⃣B LLM Backend Contract Onboarding (FLUX & MAI image backends) — continuation 5', inputs={'onboarding_params': globals().get('onboarding_params'), 'location': globals().get('location'), 'run_id': globals().get('run_id')}, expected='Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.', namespace=globals()):
    if not image_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if image_ready:
        if onboarding_params:
            deploy_contract(onboarding_params, location, "image-backends-" + run_id)
            unified_api = apimClientTool.discover_api_exact("unified-ai")


### 3️⃣ Provision Access Contract

Deploy an access contract via Bicep to create an APIM product + subscription scoped to the **enabled image models**. The product policy uses `set-llm-requested-model` (which now also resolves the `x-ai-model` header for multipart edits) and enforces `allowedModels`.

### 3️⃣ Provision Access Contract

Generate the editable parameter artifact for 3️⃣ Provision Access Contract and display the proposed configuration without deploying it.

**Input:** `contract_business_unit`, `contract_use_case_name`, `contract_environment`, `allowed_models_list`

**Output:** A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.


In [ ]:
with utils.notebook_step('3️⃣ Provision Access Contract', inputs={'contract_business_unit': globals().get('contract_business_unit'), 'contract_use_case_name': globals().get('contract_use_case_name'), 'contract_environment': globals().get('contract_environment'), 'allowed_models_list': globals().get('allowed_models_list')}, expected='A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.', namespace=globals()):
    if not image_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if image_ready:
        image_contract = {
            "name": "image-models-" + run_id, "business_unit": contract_business_unit,
            "use_case_name": contract_use_case_name, "environment": contract_environment,
            "use_keyvault": False, "endpoint_secret": "IMAGE-ENDPOINT", "apikey_secret": "IMAGE-KEY",
            "description": "Image route and multipart validation",
        }
        product_policy = access_policy(allowed_models_list, 20000, 500000, "Monthly",
            inbound_extra='<set-variable name="enableResponseHeaders" value="@(true)" />')
        utils.format_output('3️⃣ Provision Access Contract', {'product_policy': product_policy}, status='info', part='RESULT')

        image_parameters = access_parameters(context, apimClientTool, image_contract, product_policy,
            key_vault=None)

        image_params_file = write_parameters(
            repo_root / "bicep" / "infra" / "citadel-access-contracts" / "main.bicep",
            image_parameters, run_dir / "image")
        utils.format_output('3️⃣ Provision Access Contract', {'value_1': preview(image_params_file)}, status='info', part='RESULT')


### 3️⃣ Provision Access Contract — continuation 7

Deploy the reviewed artifact for 3️⃣ Provision Access Contract — continuation 7, then inspect its real resource outputs and ownership.

**Input:** `image_params_file`, `location`, `preexisting_products`, `owned_products`, `image_contract`

**Output:** Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.


In [ ]:
with utils.notebook_step('3️⃣ Provision Access Contract — continuation 7', inputs={'image_params_file': globals().get('image_params_file'), 'location': globals().get('location'), 'preexisting_products': globals().get('preexisting_products'), 'owned_products': globals().get('owned_products'), 'image_contract': globals().get('image_contract')}, expected='Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.', namespace=globals()):
    if not image_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if image_ready:
        image_outputs = deploy_contract(image_params_file, location, image_contract["name"],
            required_outputs=("products", "subscriptions"))
        image_product_id, image_subscription_name = contract_identity(
            image_outputs, preexisting_products, owned_products)
        product_id = image_product_id


### 4️⃣ Retrieve API Key & Build Endpoints

### 4️⃣ Retrieve API Key & Build Endpoints

Prepare the 4️⃣ Retrieve API Key & Build Endpoints state shown below for its subsequent feature-specific requests and assertions.

**Input:** `gateway_url`, `image_subscription_name`, `unified_api`

**Output:** `api_key`, `base_url`, `images_generations_url`, `images_edits_url`, `deployments_url`, `deployments_generations_url` are available to the following cells.


In [ ]:
with utils.notebook_step('4️⃣ Retrieve API Key & Build Endpoints', inputs={'gateway_url': globals().get('gateway_url'), 'image_subscription_name': globals().get('image_subscription_name'), 'unified_api': globals().get('unified_api')}, expected='`api_key`, `base_url`, `images_generations_url`, `images_edits_url`, `deployments_url`, `deployments_generations_url` are available to the following cells.', namespace=globals()):
    if not image_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if image_ready:
        api_key = apimClientTool.get_subscription_key(image_subscription_name)

        from urllib.parse import quote
        base_url = gateway_url
        images_generations_url = f"{unified_api['url'].rstrip('/')}/v1/images/generations"
        images_edits_url = f"{unified_api['url'].rstrip('/')}/v1/images/edits"
        deployments_url = f"{unified_api['url'].rstrip('/')}/deployments"
        def deployments_generations_url(model):
            return f"{unified_api['url'].rstrip('/')}/openai/deployments/{quote(model, safe='')}/images/generations"


### 5️⃣ Discover Deployments

Confirm the enabled image models are visible (and granted by the access contract):

### 5️⃣ Discover Deployments

Exercise 5️⃣ Discover Deployments and compare the actual response/evidence with the expected feature behavior.

**Input:** `deployments_url`, `allowed_models_list`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('5️⃣ Discover Deployments', inputs={'deployments_url': globals().get('deployments_url'), 'allowed_models_list': globals().get('allowed_models_list')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    if not image_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if image_ready:
        response = transport.request("GET", deployments_url, headers={"api-key": api_key})
        data = response.json_body if isinstance(response.json_body, dict) else {}
        names = [d.get("name") for d in data.get("value", [])]
        run.check("image-discovery", response.status_code == 200 and set(allowed_models_list) <= set(names),
            expected=allowed_models_list, actual=names, status=response.status_code)


### 6️⃣ Image Generation Helper

Posts an OpenAI-style image generation request and asserts an OpenAI-shaped response (`data[0].b64_json` or `data[0].url`).

### 6️⃣ Image Generation Helper

Exercise 6️⃣ Image Generation Helper and compare the actual response/evidence with the expected feature behavior.

**Input:** `run_dir`, `images_generations_url`, `response`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('6️⃣ Image Generation Helper', inputs={'run_dir': globals().get('run_dir'), 'images_generations_url': globals().get('images_generations_url'), 'response': globals().get('response')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    if not image_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if image_ready:
        def decoded_image(response, case):
            from PIL import Image
            import io
            response.raise_for_status()
            items = response.json().get("data", [])
            if not items:
                raise ValueError("No image data returned.")
            item = items[0]
            if item.get("b64_json"):
                image_bytes = base64.b64decode(item["b64_json"], validate=True)
            elif item.get("url"):
                # Do not forward gateway credentials to provider image URLs.
                from urllib.parse import urlparse
                if urlparse(item["url"]).scheme != "https":
                    raise ValueError("Provider image URL must use HTTPS.")
                downloaded = utils.http_call(requests.get, item["url"], timeout=120)
                downloaded.raise_for_status()
                image_bytes = downloaded.content
            else:
                raise ValueError("Image response contains neither b64_json nor url.")
            with Image.open(io.BytesIO(image_bytes)) as image:
                image.verify()
            with Image.open(io.BytesIO(image_bytes)) as image:
                if image.width < 1 or image.height < 1:
                    raise ValueError("Empty image dimensions.")
                image.load()
            path = run_dir / (re.sub(r"[^A-Za-z0-9_-]", "_", case) + ".png")
            with Image.open(io.BytesIO(image_bytes)) as image:
                image.save(path, format="PNG")
            return str(path)

        generated_image_files = {}
        def generate_image(model, prompt, extra=None, url=None, label=None):
            if model not in context.catalog.models_for("image-generation"):
                raise ValueError("Image requests require a non-skipped image-generation model.")
            label = label or model
            payload = {"model": model, "prompt": prompt, "n": 1, **(extra or {})}
            target = url or images_generations_url
            utils.format_output('6️⃣ Image Generation Helper', {'value_1': f"{label}: POST {target}; {payload}"}, status='info', part='RESULT')
            try:
                response = utils.http_call(requests.post, target, headers={"api-key": api_key}, json=payload, timeout=120)
                output = decoded_image(response, "generate-" + label)
                generated_image_files[label] = output
                run.check("generate/" + label, True, expected="Decoded nonempty image", actual=response.status_code,
                    model=model, route=target, artifact=output)
                return True
            except Exception as exc:
                run.check("generate/" + label, False, expected="Decoded nonempty image", actual=type(exc).__name__)
                return False


### 7️⃣ gpt-image Generation (body model + deployments path)

Tests both the unified `/v1/images/generations` (model in body) and the Azure OpenAI `/openai/deployments/{model}/images/generations` (model in URL) forms.

### 7️⃣ gpt-image Generation (body model + deployments path)

Prepare the 7️⃣ gpt-image Generation (body model + deployments path) state shown below for its subsequent feature-specific requests and assertions.

**Input:** `test_openai_image`, `openai_image_model`, `image_prompt`

**Output:** The visible 7️⃣ gpt-image Generation (body model + deployments path) inspection output.


In [ ]:
with utils.notebook_step('7️⃣ gpt-image Generation (body model + deployments path)', inputs={'test_openai_image': globals().get('test_openai_image'), 'openai_image_model': globals().get('openai_image_model'), 'image_prompt': globals().get('image_prompt')}, expected='The visible 7️⃣ gpt-image Generation (body model + deployments path) inspection output.', namespace=globals()):
    if not image_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if image_ready:
        if test_openai_image:
            # Body-model form (OpenAI SDK style)
            generate_image(openai_image_model, image_prompt, extra={"size": "1024x1024", "quality": "medium"}, label=f"{openai_image_model} (body)")
            # Deployments-path form (Azure OpenAI SDK style)
            generate_image(openai_image_model, image_prompt, extra={"size": "1024x1024"}, url=deployments_generations_url(openai_image_model), label=f"{openai_image_model} (deployments)")
        else:
            run.skip("openai-generation", "Provider disabled in configuration.")


### 8️⃣ FLUX Generation (azure-flux, optional)

Routes to the native Black Forest Labs surface (`/providers/blackforestlabs/v1/{modelPath}?api-version=preview`). Requires a `modelPath` slug onboarded for the model.

### 8️⃣ FLUX Generation (azure-flux, optional)

Prepare the 8️⃣ FLUX Generation (azure-flux, optional) state shown below for its subsequent feature-specific requests and assertions.

**Input:** `test_flux`, `flux_image_model`, `image_prompt`

**Output:** The visible 8️⃣ FLUX Generation (azure-flux, optional) inspection output.


In [ ]:
with utils.notebook_step('8️⃣ FLUX Generation (azure-flux, optional)', inputs={'test_flux': globals().get('test_flux'), 'flux_image_model': globals().get('flux_image_model'), 'image_prompt': globals().get('image_prompt')}, expected='The visible 8️⃣ FLUX Generation (azure-flux, optional) inspection output.', namespace=globals()):
    if not image_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if image_ready:
        if test_flux:
            generate_image(flux_image_model, image_prompt, extra={"width": 1024, "height": 1024}, label=flux_image_model)
        else:
            run.skip("flux-generation", "Provider disabled in configuration.")


### 9️⃣ MAI Generation (azure-mai, optional)

Routes to the native Microsoft MAI surface (`/mai/v1/images/generations`).

### 9️⃣ MAI Generation (azure-mai, optional)

Prepare the 9️⃣ MAI Generation (azure-mai, optional) state shown below for its subsequent feature-specific requests and assertions.

**Input:** `test_mai`, `mai_image_model`, `image_prompt`

**Output:** The visible 9️⃣ MAI Generation (azure-mai, optional) inspection output.


In [ ]:
with utils.notebook_step('9️⃣ MAI Generation (azure-mai, optional)', inputs={'test_mai': globals().get('test_mai'), 'mai_image_model': globals().get('mai_image_model'), 'image_prompt': globals().get('image_prompt')}, expected='The visible 9️⃣ MAI Generation (azure-mai, optional) inspection output.', namespace=globals()):
    if not image_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if image_ready:
        if test_mai:
            generate_image(mai_image_model, image_prompt, extra={"width": 1024, "height": 1024}, label=mai_image_model)
        else:
            run.skip("mai-generation", "Provider disabled in configuration.")


### 🔟 Missing-Model Error (multipart edits without `x-ai-model`)

`POST /unified-ai/v1/images/edits` as multipart without the `x-ai-model` header must return `400 missing_model_parameter` with guidance to set the header.

### 🔟 Missing-Model Error (multipart edits without x-ai-model)

Exercise 🔟 Missing-Model Error (multipart edits without x-ai-model) and compare the actual response/evidence with the expected feature behavior.

**Input:** `images_edits_url`, `image_prompt`, `response`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('🔟 Missing-Model Error (multipart edits without x-ai-model)', inputs={'images_edits_url': globals().get('images_edits_url'), 'image_prompt': globals().get('image_prompt'), 'response': globals().get('response')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    if not image_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if image_ready:
        # Missing x-ai-model is a distinct policy error, not a provider edit failure.
        files = {
            "prompt": (None, image_prompt),
            "image": ("blank.png", base64.b64decode("iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mNk+M9QDwADhgGAWjR9awAAAABJRU5ErkJggg=="), "image/png"),
        }
        response = transport.request("POST", images_edits_url, headers={"api-key": api_key}, files=files)
        run.check("missing-model", response.status_code == 400 and "missing_model_parameter" in response.text,
            expected="400 missing_model_parameter", actual=response.status_code)


### 🔟B Image Edit on a Generated Image (multipart)

Edits an image produced by the generation tests. Each edit prints a **summary** of the test objective — the model, the route used, the edit prompt, the **input image** filename, and the **output image** filename.

This first test uses the **deployments-path form** `POST /unified-ai/openai/deployments/{model}/images/edits` (model in the URL, so no `x-ai-model` header is needed) — the recommended form for `gpt-image`. The cells that follow add MAI and FLUX edit variations (routed via the `x-ai-model` header), and a final selectable `x-ai-model` header test.

> Run the generation cells first (with `save_images = True`) so source images exist.

### 🔟B Image Edit on a Generated Image (multipart)

Exercise 🔟B Image Edit on a Generated Image (multipart) and compare the actual response/evidence with the expected feature behavior.

**Input:** `test_openai_image`, `images_edits_url`, `output`, `openai_image_model`, `target`, `response`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('🔟B Image Edit on a Generated Image (multipart)', inputs={'test_openai_image': globals().get('test_openai_image'), 'images_edits_url': globals().get('images_edits_url'), 'output': globals().get('output'), 'openai_image_model': globals().get('openai_image_model'), 'target': globals().get('target'), 'response': globals().get('response')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    if not image_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if image_ready:
        def edit_image(model, image_path, prompt, use_header=False, size=None):
            case = f"edit/{model}/{'header' if use_header else 'path'}"
            if model not in context.catalog.models_for("image-edit"):
                if use_header and model != edit_header_model:
                    run.skip(case, "Optional provider lacks explicit image-edit capability; generation does not imply edits.")
                else:
                    run.skip(case, "No eligible image-edit operation; generation coverage does not require edits.")
                return None
            if not image_path or not Path(image_path).is_file():
                run.blocked(case, "Generation must first produce a decoded input image.")
                return None
            target = images_edits_url if use_header else f"{unified_api['url'].rstrip('/')}/openai/deployments/{quote(model, safe='')}/images/edits"
            headers = {"api-key": api_key}
            if use_header:
                headers["x-ai-model"] = model
            form = {"prompt": (None, prompt)}
            if size:
                form["size"] = (None, size)
            try:
                with open(image_path, "rb") as image:
                    files = {**form, "image": (Path(image_path).name, image.read(), "image/png")}
                # requests supplies multipart Content-Type and boundary; do not set JSON headers.
                response = utils.http_call(requests.post, target, headers=headers, files=files, timeout=180)
                output = decoded_image(response, case)
                run.check(case, True, expected="Decoded edited image", actual=response.status_code,
                    model=model, route=target, artifact=output)
                return output
            except Exception as exc:
                run.check(case, False, expected="Decoded edited image", actual=type(exc).__name__)
                return None

        def summarize_edit(title, model, route, prompt, input_path, output_path):
            """Print a concise summary of an edit test: objective, input image, output image."""
            utils.print_info("────────────────────────────────────────────────────────")
            utils.print_info(f"🧪 {title}")
            utils.print_info(f"   Model        : {model}")
            utils.print_info(f"   Route        : {route}")
            utils.print_info(f"   Edit prompt  : {prompt}")
            utils.print_info(f"   Input image  : {os.path.basename(input_path) if input_path else '(none — generate first)'}")
            if output_path:
                utils.print_ok(f"   Output image : {os.path.basename(output_path)} ✅")
            else:
                utils.print_warning("   Output image : (edit did not produce an image)")
            utils.print_info("────────────────────────────────────────────────────────")


        def _source_image_for(model):
            """Pick a previously generated image to edit, preferring one made by `model`."""
            return (generated_image_files.get(model)
                    or generated_image_files.get(f"{model} (body)")
                    or generated_image_files.get(f"{openai_image_model} (body)")
                    or generated_image_files.get(f"{openai_image_model} (deployments)")
                    or next(iter(generated_image_files.values()), None))


        # --- gpt-image edit via the deployments path (model in URL) ---
        edit_prompt = "Add a light snowfall and a soft blue winter tone to the scene"
        if test_openai_image:
            src = _source_image_for(openai_image_model)
            out_path = edit_image(openai_image_model, src, edit_prompt, size="1024x1024")
            summarize_edit(
                title="gpt-image edit — deployments path (model in URL)",
                model=openai_image_model,
                route=f"POST /unified-ai/openai/deployments/{openai_image_model}/images/edits",
                prompt=edit_prompt,
                input_path=src,
                output_path=out_path,
            )
        else:
            run.skip("openai-edit", "Provider disabled in configuration.")


### 🔟C MAI Edit (different variation, `x-ai-model` header)

Edits a generated image with **MAI** via `POST /unified-ai/v1/images/edits` and the `x-ai-model` header (MAI routes natively to `/mai/v1/images/edits`, where the model can't ride the URL). Uses a different edit prompt — a watercolor/sunset restyle — and prints the same input/output summary.

### 🔟C MAI Edit (different variation, x-ai-model header)

Prepare the 🔟C MAI Edit (different variation, x-ai-model header) state shown below for its subsequent feature-specific requests and assertions.

**Input:** `test_mai`, `mai_image_model`, `src`, `out_path`

**Output:** `mai_edit_prompt`, `src`, `out_path` are available to the following cells.


In [ ]:
with utils.notebook_step('🔟C MAI Edit (different variation, x-ai-model header)', inputs={'test_mai': globals().get('test_mai'), 'mai_image_model': globals().get('mai_image_model'), 'src': globals().get('src'), 'out_path': globals().get('out_path')}, expected='`mai_edit_prompt`, `src`, `out_path` are available to the following cells.', namespace=globals()):
    if not image_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if image_ready:
        # MAI edit — routed via /v1/images/edits + x-ai-model header (native /mai/v1/images/edits)
        mai_edit_prompt = "Restyle the scene as a vibrant watercolor painting with warm sunset colors"
        if test_mai:
            src = _source_image_for(mai_image_model)
            out_path = edit_image(mai_image_model, src, mai_edit_prompt, use_header=True)
            summarize_edit(
                title="MAI edit — x-ai-model header (native /mai/v1/images/edits)",
                model=mai_image_model,
                route="POST /unified-ai/v1/images/edits  (x-ai-model: " + mai_image_model + ")",
                prompt=mai_edit_prompt,
                input_path=src,
                output_path=out_path,
            )
        else:
            run.skip("mai-edit", "Provider disabled in configuration.")


### 🔟D FLUX Edit (different variation, `x-ai-model` header)

Edits a generated image with **FLUX** via `POST /unified-ai/v1/images/edits` and the `x-ai-model` header (FLUX routes natively to `/providers/blackforestlabs/v1/{modelPath}`). Uses a third edit prompt — a night-time bioluminescent restyle.

> Edit support varies by FLUX model (e.g. `FLUX.1-Kontext-pro` is edit-oriented). If your configured FLUX model doesn't accept the multipart edit, the summary will report no output and the response body explains why.

### 🔟D FLUX Edit (different variation, x-ai-model header)

Prepare the 🔟D FLUX Edit (different variation, x-ai-model header) state shown below for its subsequent feature-specific requests and assertions.

**Input:** `test_flux`, `flux_image_model`, `src`, `out_path`

**Output:** `flux_edit_prompt`, `src`, `out_path` are available to the following cells.


In [ ]:
with utils.notebook_step('🔟D FLUX Edit (different variation, x-ai-model header)', inputs={'test_flux': globals().get('test_flux'), 'flux_image_model': globals().get('flux_image_model'), 'src': globals().get('src'), 'out_path': globals().get('out_path')}, expected='`flux_edit_prompt`, `src`, `out_path` are available to the following cells.', namespace=globals()):
    if not image_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if image_ready:
        # FLUX edit — routed via /v1/images/edits + x-ai-model header (native BFL surface)
        flux_edit_prompt = "Reimagine the scene at night with glowing bioluminescent plants and a starry sky"
        if test_flux:
            src = _source_image_for(flux_image_model)
            out_path = edit_image(flux_image_model, src, flux_edit_prompt, use_header=True)
            summarize_edit(
                title="FLUX edit — x-ai-model header (native BFL surface)",
                model=flux_image_model,
                route="POST /unified-ai/v1/images/edits  (x-ai-model: " + flux_image_model + ")",
                prompt=flux_edit_prompt,
                input_path=src,
                output_path=out_path,
            )
        else:
            run.skip("flux-edit", "Provider disabled in configuration.")


### 🔟E Edit via `x-ai-model` Header (selectable model)

A generic, **selectable** edit test that routes purely through the unified `POST /unified-ai/v1/images/edits` surface using the `x-ai-model` header. Set `edit_header_model` to any onboarded image model whose backend supports edits (gpt-image, MAI, or — where supported — FLUX) and run the cell. This is the form to use when the client can't (or doesn't want to) put the model in the URL.

### 🔟E Edit via x-ai-model Header (selectable model)

Prepare the 🔟E Edit via x-ai-model Header (selectable model) state shown below for its subsequent feature-specific requests and assertions.

**Input:** `edit_header_model`, `src`, `out_path`

**Output:** `edit_header_prompt`, `src`, `out_path` are available to the following cells.


In [ ]:
with utils.notebook_step('🔟E Edit via x-ai-model Header (selectable model)', inputs={'edit_header_model': globals().get('edit_header_model'), 'src': globals().get('src'), 'out_path': globals().get('out_path')}, expected='`edit_header_prompt`, `src`, `out_path` are available to the following cells.', namespace=globals()):
    if not image_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if image_ready:
        if edit_header_model is not None:
            # ── Selectable target model for the x-ai-model header edit ──────────────────
            # Change this to flux_image_model or mai_image_model to target those backends.
            # edit_header_model is capability-selected in preflight.
            edit_header_prompt = "Add a subtle vintage film grain with warm sepia tones"

            src = _source_image_for(edit_header_model)
            out_path = edit_image(edit_header_model, src, edit_header_prompt, use_header=True)
            summarize_edit(
                title="x-ai-model header edit (selectable model)",
                model=edit_header_model,
                route="POST /unified-ai/v1/images/edits  (x-ai-model: " + str(edit_header_model) + ")",
                prompt=edit_header_prompt,
                input_path=src,
                output_path=out_path,
            )
        else:
            run.skip("image-edit-header", "No eligible image-edit model.")


### 1️⃣1️⃣ RBAC Enforcement (disallowed model → 403)

Requesting an image model that is NOT in the access contract's `allowedModels` must be rejected.

### 1️⃣1️⃣ RBAC Enforcement (disallowed model → 403)

Exercise 1️⃣1️⃣ RBAC Enforcement (disallowed model → 403) and compare the actual response/evidence with the expected feature behavior.

**Input:** `disallowed_image_model`, `image_prompt`, `images_generations_url`, `payload`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('1️⃣1️⃣ RBAC Enforcement (disallowed model → 403)', inputs={'disallowed_image_model': globals().get('disallowed_image_model'), 'image_prompt': globals().get('image_prompt'), 'images_generations_url': globals().get('images_generations_url'), 'payload': globals().get('payload')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    if not image_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if image_ready:
        if not disallowed_image_model:
            run.skip("image-rbac", "No eligible image-generation model remains outside the contract grants.")
        else:
            payload = {"model": disallowed_image_model, "prompt": image_prompt, "n": 1}
            response = transport.request("POST", images_generations_url, headers={"api-key": api_key}, json=payload)
            run.check("image-rbac", response.status_code == 403 and "unauthorized_model_access" in response.text,
                expected="403 unauthorized_model_access", actual=response.status_code, model=disallowed_image_model)


### 1️⃣1️⃣ RBAC Enforcement (disallowed model → 403) — continuation 20

Prepare the 1️⃣1️⃣ RBAC Enforcement (disallowed model → 403) — continuation 20 state shown below for its subsequent feature-specific requests and assertions.

**Input:** `owned_products`, `cleanup_delete_contract`

**Output:** The visible 1️⃣1️⃣ RBAC Enforcement (disallowed model → 403) — continuation 20 inspection output.


In [ ]:
with utils.notebook_step('1️⃣1️⃣ RBAC Enforcement (disallowed model → 403) — continuation 20', inputs={'owned_products': globals().get('owned_products'), 'cleanup_delete_contract': globals().get('cleanup_delete_contract')}, expected='The visible 1️⃣1️⃣ RBAC Enforcement (disallowed model → 403) — continuation 20 inspection output.', namespace=globals()):
    if not image_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if image_ready:
        cleanup_products(apimClientTool, owned_products, enabled=cleanup_delete_contract)


### 1️⃣1️⃣ RBAC Enforcement (disallowed model → 403) — continuation 21

Collect the final 1️⃣1️⃣ RBAC Enforcement (disallowed model → 403) — continuation 21 verdict without discarding earlier failed or blocked required cases.

**Input:** The initialized configuration and the explicit literals in this cell.

**Output:** The strict validation summary; any failed or blocked required coverage raises.


In [ ]:
with utils.notebook_step('1️⃣1️⃣ RBAC Enforcement (disallowed model → 403) — continuation 21', inputs={}, expected='The strict validation summary; any failed or blocked required coverage raises.', namespace=globals()):
    if not image_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if image_ready:
        run.finish()
    else:
        utils.format_output("Image scenario", {"reason": "No eligible image models; no tests executed."}, status="SKIP", part="RESULT")


## ✅ Summary

This notebook validated image generation/editing through the Unified AI API:
- Onboarded dedicated `azure-flux` / `azure-mai` image backends (gpt-image rides the existing `ai-foundry` backend)
- gpt-image generation via body-model and deployments-path forms
- FLUX (`azure-flux`) and MAI (`azure-mai`) native surfaces (when enabled)
- A real image edit on a generated image via the deployments-path multipart form
- Multipart-edit missing-model guidance (`x-ai-model` header requirement)
- RBAC enforcement via the access contract's `allowedModels`

See [LLM Access Guide — Image Models](../guides/llm-access-guide.md#image-models) and [LLM Backend Onboarding — Image Models](../bicep/infra/llm-backend-onboarding/README.md#image-models) for details.